In [0]:
%sql
WITH m AS (SELECT * FROM laddstolpar_df.gold.mart_kommun_metric),
w AS (
  SELECT a.kommun_kod,
         a.pct_rank    AS g_shrunk,
         r.pct_rank    AS g_raw,
         d.pct_rank    AS demand,
         a.denominator AS n_new_cars,
         a.value       AS d_shrunk,
         r.value       AS d_raw
  FROM m a
  JOIN m r USING (kommun_kod)
  JOIN m d USING (kommun_kod)
  WHERE a.metric = 'growth' AND r.metric = 'growth_raw' AND d.metric = 'ev_demand'
)
SELECT
  ROUND(corr(g_raw, demand), 2)                         AS rho_raw_demand,
  ROUND(corr(g_shrunk, demand), 2)                      AS rho_shrunk_demand,
  ROUND(corr(g_shrunk, g_raw), 2)                       AS rho_shrunk_raw,
  CAST(percentile(n_new_cars, 0.5) AS BIGINT)           AS median_new_cars,
  COUNT_IF(n_new_cars < 46)                             AS kommuner_below_k,
  ROUND(100 * AVG(ABS(d_shrunk - d_raw)), 2)            AS mean_shift_pp,
  ROUND(100 * MAX(ABS(d_shrunk - d_raw)), 2)            AS max_shift_pp
FROM w;

In [0]:
%sql
WITH a AS (
  SELECT a.*, k.kommun_namn
  FROM laddstolpar_df.gold.mart_allocation a
  JOIN laddstolpar_df.gold.dim_kommun k USING (kommun_kod)
),
m AS (SELECT * FROM laddstolpar_df.gold.mart_kommun_metric)
SELECT
  (SELECT COUNT(*) FROM laddstolpar_df.gold.mart_kommun_score_component)                   AS component_rows,
  (SELECT COUNT(*) FROM laddstolpar_df.gold.mart_kommun_score)                             AS score_rows,
  (SELECT COUNT(DISTINCT scenario, variant) FROM a)                                        AS runs,
  (SELECT CAST(MIN(t) AS STRING) || '-' || CAST(MAX(t) AS STRING)
     FROM (SELECT SUM(n_stations) AS t FROM a GROUP BY scenario, variant))                 AS stations_per_run,
  (SELECT MAX(n_stations) FROM a)                                                          AS max_in_one_kommun,
  (SELECT concat_ws(', ', transform(array_sort(collect_list(struct(-n_stations AS s, score_rank AS r,
            kommun_namn || ' ' || CAST(n_stations AS STRING) AS t))), x -> x.t))
     FROM a WHERE scenario = 'S1' AND variant = 'base' AND n_stations > 0)                 AS s1_base,
  (SELECT concat_ws(', ', transform(array_sort(collect_list(struct(-n_stations AS s, score_rank AS r,
            kommun_namn || ' ' || CAST(n_stations AS STRING) AS t))), x -> x.t))
     FROM a WHERE scenario = 'S2' AND variant = 'base' AND n_stations > 0)                 AS s2_base,
  (SELECT concat_ws(', ', transform(array_sort(collect_list(struct(-n_stations AS s, score_rank AS r,
            kommun_namn || ' ' || CAST(n_stations AS STRING) AS t))), x -> x.t))
     FROM a WHERE scenario = 'S3' AND variant = 'base' AND n_stations > 0)                 AS s3_base,
  (SELECT COUNT(*) FROM (SELECT kommun_kod FROM a WHERE variant = 'base' AND n_stations > 0
     GROUP BY kommun_kod HAVING COUNT(*) = 3))                                             AS robust_in_all_3,
  (SELECT ROUND(corr(x.pct_rank, y.pct_rank), 2) FROM m x JOIN m y USING (kommun_kod)
     WHERE x.metric = 'growth_stock' AND y.metric = 'ev_demand')                           AS rho_c_demand,
  (SELECT ROUND(corr(x.pct_rank, y.pct_rank), 2) FROM m x JOIN m y USING (kommun_kod)
     WHERE x.metric = 'growth' AND y.metric = 'growth_stock')                              AS rho_d_c;

In [0]:
%sql
WITH a AS (
  SELECT * FROM laddstolpar_df.gold.mart_allocation WHERE variant IN ('base', 'no_cost')
),
thr AS (
  SELECT scenario, variant, MIN(quotient) AS q50
  FROM laddstolpar_df.gold.mart_allocation_station
  GROUP BY scenario, variant
)
SELECT a.scenario, a.variant,
       ROUND(MAX(t.q50), 3)                                              AS quotient_50th,
       ROUND(MAX(a.score) / 3, 3)                                        AS best_possible_2nd_station,
       ROUND(MAX(CASE WHEN a.kommun_kod = '0180' THEN a.score END), 3)   AS sthlm_score,
       MAX(CASE WHEN a.kommun_kod = '0180' THEN a.score_rank END)        AS sthlm_rank,
       MAX(CASE WHEN a.kommun_kod = '0180' THEN a.n_stations END)        AS sthlm_n,
       MAX(CASE WHEN a.kommun_kod = '1480' THEN a.score_rank END)        AS gbg_rank,
       MAX(CASE WHEN a.kommun_kod = '1480' THEN a.n_stations END)        AS gbg_n,
       MAX(CASE WHEN a.kommun_kod = '1280' THEN a.score_rank END)        AS malmo_rank,
       MAX(CASE WHEN a.kommun_kod = '1280' THEN a.n_stations END)        AS malmo_n
FROM a
JOIN thr t USING (scenario, variant)
GROUP BY a.scenario, a.variant
ORDER BY a.scenario, a.variant;